In Class Assignment 2026.04.07
(3 points) Using pandas, load the dataset and display the first 5 rows.

In [4]:
!pip install sweetviz

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/15.1 MB ? eta -:--:--
   -- ------------------------------------- 0.8/15.1 MB 8.2 MB/s eta 0:00:02
   ----------- ---------------------------- 4.2/15.1 MB 13.6 MB/s eta 0:00:01
   ---------------------- ----------------- 8.4/15.1 MB 16.5 MB/s eta 0:00:01
   --------------------------------- ------ 12.8/15.1 MB 18.2 MB/s eta 0:00:01
   ---------------------------------------- 15.1/15.1 MB 17.0 MB/s  0:00:01
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)

   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0/3 [tqdm]
   ---------------------------------------- 0


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import numpy as np
import sweetviz as sv

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

insurance = pd.read_csv("C:/Users/2003n/Downloads/insurance.csv")
insurance.head(10)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


In [9]:
report = sv.analyze(insurance)
report.show_html("insurance_report.html")

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [ ]:
#encoding categorical variables
insurance_encoded = pd.get_dummies(insurance, drop_first = True).astype(int)
insurance_encoded.head(10)

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0
5,31,25,0,3756,0,0,0,1,0
6,46,33,1,8240,0,0,0,1,0
7,37,27,3,7281,0,0,1,0,0
8,37,29,2,6406,1,0,0,0,0
9,60,25,0,28923,0,0,1,0,0


In [15]:
X = insurance_encoded.drop('charges',axis = 1)
y = insurance_encoded['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = 0.2, random_state = 42)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#baseline linear regression model
base_lr = LinearRegression()
base_lr.fit(X_train_scaled, y_train)
y_pred_lr = base_lr.predict(X_test_scaled)
mse_lr= mean_squared_error (y_test, y_pred_lr)
r2_lr = r2_score(y_test,y_pred_lr)

#baseline random forest model
base_rf = RandomForestRegressor(random_state=42)
base_rf.fit(X_train_scaled, y_train)
y_pred_rf = base_rf.predict(X_test_scaled)
mse_rf= mean_squared_error (y_test, y_pred_rf)
r2_rf = r2_score(y_test,y_pred_rf)

print("Linear Regression Results")
print("R2:", r2_lr)
print("MSE:", mse_lr)

print("\nRandom Forest Results")
print("R2:", r2_rf)
print("MSE:", mse_rf)

Linear Regression Results
R2: 0.7837888448800692
MSE: 33566439.73530044

Random Forest Results
R2: 0.8575260190344218
MSE: 22118860.11744718


Linear: Shows a good baseline, but the performance may tank depending on whether or not the variables are not purely linear.
Random Forest: Good performance, especially because it can deal with nonlinear relationships.

Overall, the random forest was the better model due to the higher R2 value, which explains about 85.8% of the data, and a lower MSE.

In [16]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Parameters: {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 300}
Best CV Score: 0.8421399151526435


The best parameters from GridSearchCV were:

- max_depth = None
- max_features = 'log2'
- min_samples_split = 5
- n_estimators = 300

The best cross-validation R2 score was 0.8421.

In [17]:
cv_results = pd.DataFrame(grid_search.cv_results_)

top_20_models = cv_results.sort_values(by='rank_test_score').head(20)

columns_to_show = [
    'rank_test_score',
    'mean_test_score',
    'std_test_score',
    'mean_train_score',
    'param_n_estimators',
    'param_max_depth',
    'param_min_samples_split',
    'param_max_features'
]

top_20_models[columns_to_show].reset_index(drop=True)

,rank_test_score,mean_test_score,std_test_score,mean_train_score,param_n_estimators,param_max_depth,param_min_samples_split,param_max_features
0,1,0.842140,0.039483,0.940085,300,None,5,log2
1,1,0.842140,0.039483,0.940085,300,20,5,log2
2,1,0.842140,0.039483,0.940085,300,30,5,log2
3,4,0.842025,0.040510,0.933498,300,10,5,log2
4,5,0.841455,0.040042,0.932872,200,10,5,log2
5,6,0.841202,0.038886,0.939382,200,30,5,log2
6,6,0.841202,0.038886,0.939382,200,20,5,log2
7,6,0.841202,0.038886,0.939382,200,None,5,log2
8,9,0.841059,0.040161,0.959376,300,10,2,log2
9,10,0.840957,0.040571,0.910406,300,None,10,log2


The best parameters for the top models were n_estimators = 30, max_depth = 20, min_samples_split = 5, and max_features = log2. While other parameterse vary from the top top models, the mean_test_score does not change significantly or at all. So these 4 are the main most consistent hyperparameters. 

In [19]:
best_rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    max_features='log2',
    random_state=42
)

best_rf.fit(X_train_scaled, y_train)

y_train_pred = best_rf.predict(X_train_scaled)
y_test_pred = best_rf.predict(X_test_scaled)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print("Train R2:", train_r2)
print("Test R2:", test_r2)
print("Train MSE:", train_mse)
print("Test MSE:", test_mse)

Train R2: 0.9397827159393439
Test R2: 0.8663152126010755
Train MSE: 8691386.219540594
Test MSE: 20754351.722802483


The model seems to perform better on the training set than the test set, which most likely indicates that there is some overfitting happening. This can be seen with the higher training R2 and a significantly lower MSE than the test results. Although there is overfitting, the test performance is still good at generalization of unseen data. 